# Modelo de Clasificacion de Plagio

Este notebook entrena un modelo simple con TensorFlow para decidir si un **par de archivos de codigo** es plagio (1) o no plagio (0).

El modelo no lee codigo directamente: recibe un **vector de caracteristicas** con las metricas de similitud (similitud de Winnowing, cobertura de fragmentos compartidos, etc.) que generara el parser de AST.

Mientras ese parser no existe, usamos datos sinteticos para probar que el modelo funciona.

## 1. Importar librerias

Cargamos TensorFlow para la red neuronal y scikit-learn para dividir los datos y normalizarlos.

In [1]:
import subprocess
result = subprocess.run(
    ["python", "-c", "import cpuinfo; print(cpuinfo.get_cpu_info()['flags'])"],
    capture_output=True, text=True
)
print(result.stdout)

In [2]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler

## 2. Configuracion

Definimos una semilla para que los resultados sean reproducibles, las columnas de caracteristicas que espera el modelo y el nombre de la columna con la etiqueta.

In [3]:
RANDOM_SEED = 42
FEATURE_COLUMNS = [
    "winnowing_similarity",
    "shared_fragment_coverage",
    "token_overlap_ratio",
    "ast_depth_difference",
    "fingerprint_jaccard",
]
LABEL_COLUMN = "label"

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

## 3. Cargar los datos

Si existe un archivo CSV con las caracteristicas ya calculadas, lo usamos. Si no, generamos datos sinteticos: los pares de plagio tienen valores de similitud altos y los de no plagio valores bajos.

In [4]:
def load_dataset(csv_path):
    df = pd.read_csv(csv_path)
    x = df[FEATURE_COLUMNS].values.astype("float32")
    y = df[LABEL_COLUMN].values.astype("float32")
    return x, y


def generate_synthetic_dataset(n_samples=2000, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    n_pos = n_samples // 2
    n_neg = n_samples - n_pos
    pos = rng.normal(loc=0.8, scale=0.12, size=(n_pos, len(FEATURE_COLUMNS)))
    neg = rng.normal(loc=0.25, scale=0.12, size=(n_neg, len(FEATURE_COLUMNS)))
    x = np.clip(np.vstack([pos, neg]), 0.0, 1.0).astype("float32")
    y = np.concatenate([np.ones(n_pos), np.zeros(n_neg)]).astype("float32")
    idx = rng.permutation(n_samples)
    return x[idx], y[idx]


csv_path = "pairs_features.csv"
if os.path.exists(csv_path):
    print(f"Loading dataset from {csv_path}")
    x, y = load_dataset(csv_path)
else:
    print("Dataset not found, using synthetic data")
    x, y = generate_synthetic_dataset()

print(x.shape, y.shape)

Loading dataset from pairs_features.csv
(1900, 5) (1900,)


## 4. Construir el modelo

Es una red neuronal pequena: dos capas ocultas y una salida con `sigmoid` que da una probabilidad entre 0 y 1.

Medimos las metricas que pide el proyecto: precision, recall y AUC-ROC.

In [5]:
def build_model(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="auc"),
        ],
    )
    return model


def f1_from(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


build_model(len(FEATURE_COLUMNS)).summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 737 (2.88 KB)

 Trainable params: 737 (2.88 KB)

 Non-trainable params: 0 (0.00 B)

## 5. Validacion cruzada (5-fold)

Dividimos los datos en 5 partes y entrenamos 5 veces, usando cada parte como validacion una vez. Asi vemos un desempeno mas confiable que con una sola division.

Antes de entrenar, normalizamos las caracteristicas para que todas esten en una escala parecida.

In [6]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(x, y), start=1):
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x[train_idx])
    x_val = scaler.transform(x[val_idx])
    model = build_model(x.shape[1])
    model.fit(
        x_train, y[train_idx],
        validation_data=(x_val, y[val_idx]),
        epochs=30,
        batch_size=32,
        verbose=0,
    )
    metrics = model.evaluate(x_val, y[val_idx], verbose=0, return_dict=True)
    metrics["f1"] = f1_from(metrics["precision"], metrics["recall"])
    scores.append(metrics)
    print(
        f"Fold {fold}: "
        f"acc={metrics['accuracy']:.3f} "
        f"precision={metrics['precision']:.3f} "
        f"recall={metrics['recall']:.3f} "
        f"f1={metrics['f1']:.3f} "
        f"auc={metrics['auc']:.3f}"
    )

Fold 1: acc=0.526 precision=0.553 recall=0.274 f1=0.366 auc=0.557
Fold 2: acc=0.534 precision=0.842 recall=0.084 f1=0.153 auc=0.573
Fold 3: acc=0.563 precision=0.962 recall=0.132 f1=0.231 auc=0.585
Fold 4: acc=0.563 precision=0.650 recall=0.274 f1=0.385 auc=0.597
Fold 5: acc=0.497 precision=0.498 recall=0.532 f1=0.514 auc=0.527


## 6. Resumen de la validacion cruzada

Promediamos las metricas de los 5 folds para tener un numero final por metrica.

In [7]:
for key in ["accuracy", "precision", "recall", "f1", "auc"]:
    values = np.array([s[key] for s in scores])
    print(f"{key}: {values.mean():.3f} +/- {values.std():.3f}")

accuracy: 0.537 +/- 0.025
precision: 0.701 +/- 0.175
recall: 0.259 +/- 0.156
f1: 0.330 +/- 0.126
auc: 0.568 +/- 0.024


## 7. Modelo final y prueba

Entrenamos un modelo final con el 80% de los datos y lo evaluamos en el 20% restante, que nunca vio durante el entrenamiento. Estas son las metricas que comparamos contra la linea base de Dolos (F1 = 0.865).

In [8]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

model = build_model(x.shape[1])
model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32,
    verbose=0,
)

metrics = model.evaluate(x_test, y_test, verbose=0, return_dict=True)
metrics["f1"] = f1_from(metrics["precision"], metrics["recall"])
print(
    f"acc={metrics['accuracy']:.3f} "
    f"precision={metrics['precision']:.3f} "
    f"recall={metrics['recall']:.3f} "
    f"f1={metrics['f1']:.3f} "
    f"auc={metrics['auc']:.3f}"
)

acc=0.553 precision=0.955 recall=0.111 f1=0.198 auc=0.534


## 8. Guardar el modelo

Guardamos el modelo entrenado para poder reutilizarlo despues sin volver a entrenar.

In [9]:
model.save("plagiarism_model.keras")
print("Modelo guardado en plagiarism_model.keras")

Modelo guardado en plagiarism_model.keras
